In [ ]:
%%bash
nproc; free -g | head -2
df -h /tmp /kaggle/working

In [ ]:
%%bash
set -e
LOG=/tmp/pg_build.log; : > $LOG
run() { "$@" >> $LOG 2>&1 || { echo "LỖI khi chạy: $*"; tail -30 $LOG; exit 1; }; }

# Mã nguồn của tác giả: Bao (server + extension), script nạp IMDB, các split
[ -d /tmp/lqo_ml_perspective ] || git clone -q https://github.com/edualc/lqo_ml_perspective.git /tmp/lqo_ml_perspective

PKG=""
[ -x /opt/pg12/bin/pg_config ] && PKG=$(/opt/pg12/bin/pg_config --pkglibdir)
if [ -n "$PKG" ] && [ -f "$PKG/pg_bao.so" ] && [ -f "$PKG/pg_prewarm.so" ]; then
  echo "Đã build từ trước, bỏ qua."
else
  run apt-get update -qq
  run apt-get install -y -qq build-essential libreadline-dev zlib1g-dev
  cd /tmp
  rm -rf postgresql-12.5 postgresql-12.5.tar.gz
  run wget -q https://ftp.postgresql.org/pub/source/v12.5/postgresql-12.5.tar.gz
  tar xzf postgresql-12.5.tar.gz && cd postgresql-12.5
  run ./configure --prefix=/opt/pg12
  run make -j"$(nproc)"
  run make install
  run make -C contrib/pg_prewarm install          # để làm nóng cache
  # Extension của Bao (giống infra/Dockerfile.pg trên Mac)
  rm -rf /tmp/pg_bao && cp -r /tmp/lqo_ml_perspective/bao/pg_extension /tmp/pg_bao
  run make -C /tmp/pg_bao USE_PGXS=1 PG_CONFIG=/opt/pg12/bin/pg_config install
fi
/opt/pg12/bin/postgres --version
PKG=$(/opt/pg12/bin/pg_config --pkglibdir)
[ -f "$PKG/pg_bao.so" ] || { echo "Không thấy pg_bao.so trong $PKG"; exit 1; }
echo "pg_bao OK"

In [ ]:
%%bash
set -e
id -u postgres > /dev/null 2>&1 || useradd -m -s /bin/bash postgres
if [ ! -f /tmp/pgdata/PG_VERSION ]; then
mkdir -p /tmp/pgdata && chown postgres /tmp/pgdata
su postgres -c "/opt/pg12/bin/initdb -D /tmp/pgdata -U postgres --auth=trust" > /dev/null
# Bộ nhớ GIỐNG notebook HybridQO để so hai phương pháp trên cùng môi trường.
# Khác HybridQO: geqo = on (chú thích ¹ Bảng 2: GEQO bật cho Bao và PostgreSQL), nạp pg_bao.
cat >> /tmp/pgdata/postgresql.conf <<'EOF'
listen_addresses = 'localhost'
shared_buffers = 8GB
temp_buffers = 32GB
work_mem = 512MB
effective_cache_size = 32GB
max_parallel_workers_per_gather = 4
max_parallel_maintenance_workers = 4
geqo = on
autovacuum = off
shared_preload_libraries = 'pg_bao'
dynamic_shared_memory_type = mmap
EOF
fi
su postgres -c "/opt/pg12/bin/pg_ctl -D /tmp/pgdata status" > /dev/null \
  || su postgres -c "/opt/pg12/bin/pg_ctl -D /tmp/pgdata -l /tmp/pg.log start -w"
/opt/pg12/bin/psql -U postgres -h localhost -Atc "SELECT name, setting FROM pg_settings WHERE name IN ('shared_buffers','work_mem','geqo','shared_preload_libraries')"

In [ ]:
%%bash
set -e
export PATH=/opt/pg12/bin:$PATH PGHOST=localhost PGUSER=postgres
[ -x /opt/pg12/bin/psql ] || { echo "Chưa có PostgreSQL (phiên Kaggle mới?): chạy lại ô 1 và 2."; exit 1; }
pg_isready -q || { echo "PostgreSQL chưa chạy: chạy lại ô 2."; exit 1; }
if pgrep -f load_job_postgres.sh > /dev/null; then
  echo "Lần chạy trước vẫn đang nạp. Xem: tail /tmp/load.log. Đợi xong rồi chạy lại ô này."; exit 1
fi
if [ -f /tmp/imdb_loaded ]; then
  echo "IMDB đã nạp từ trước, bỏ qua."
else
  dropdb --if-exists imdbload
  rm -rf /tmp/imdb && mkdir -p /tmp/imdb
  wget -q -c -O /tmp/imdb.tgz https://event.cwi.nl/da/job/imdb.tgz
  tar xzf /tmp/imdb.tgz -C /tmp/imdb
  pip install -q absl-py
  python /tmp/lqo_ml_perspective/data/prepend_imdb_headers.py --csv_dir /tmp/imdb 2> /dev/null
  chmod -R a+rX /tmp/imdb /tmp/lqo_ml_perspective
  echo "Đang nạp, khoảng 30–45 phút; log ở /tmp/load.log ..."
  su postgres -c "PATH=/opt/pg12/bin:\$PATH PGHOST=localhost bash /tmp/lqo_ml_perspective/balsa/load-postgres/load_job_postgres.sh /tmp/imdb" > /tmp/load.log 2>&1 &
  PID=$!
  while kill -0 $PID 2> /dev/null; do
    sleep 60
    N=$(grep -c '^COPY ' /tmp/load.log || true)
    if grep -q 'ANALYZE VERBOSE' /tmp/load.log; then STEP="ANALYZE (bước cuối)"
    elif grep -q add_fks /tmp/load.log; then STEP="tạo khóa ngoại"
    else STEP="nạp CSV"; fi
    echo "$(date +%H:%M)  đã nạp $N/21 bảng, đang: $STEP"
  done
  wait $PID || { echo "Script nạp dừng vì lỗi:"; tail -20 /tmp/load.log; exit 1; }
  if grep -q ERROR /tmp/load.log; then echo "Có lỗi khi nạp:"; grep ERROR /tmp/load.log | head -10; exit 1; fi
fi
BAD=""
while read -r t n; do
  c=$(psql -d imdbload -At -v ON_ERROR_STOP=1 -c "SELECT count(*) FROM $t" < /dev/null) \
    || { echo "Không đếm được bảng $t"; exit 1; }
  [ "$c" = "$n" ] || BAD+="$t: có $c, cần $n"$'\n'
done <<'EOF'
aka_name 901343
aka_title 361472
cast_info 36244344
char_name 3140339
comp_cast_type 4
company_name 234997
company_type 4
complete_cast 135086
info_type 113
keyword 134170
kind_type 7
link_type 18
movie_companies 2609129
movie_info 14835720
movie_info_idx 1380035
movie_keyword 4523930
movie_link 29997
name 4167491
person_info 2963664
role_type 12
title 2528312
EOF
[ -z "$BAD" ] || { echo "Các bảng sau sai số dòng:"; echo "$BAD"; exit 1; }
echo "Đủ 21 bảng, số dòng khớp."
touch /tmp/imdb_loaded
rm -rf /tmp/imdb /tmp/imdb.tgz

In [ ]:
%%bash
set -e
pip install -q uv
[ -x /tmp/bao_env/bin/python ] || uv venv -q -p 3.10 /tmp/bao_env
# Giống lqo_ml_perspective/bao/requirements.txt (torch bản CPU)
uv pip install -q --python /tmp/bao_env/bin/python torch==1.13.1 --index-url https://download.pytorch.org/whl/cpu
uv pip install -q --python /tmp/bao_env/bin/python numpy==1.23.5 scikit-learn==1.1.3 scipy==1.9.3 joblib==1.2.0 \
  threadpoolctl==3.1.0 psycopg2-binary==2.9.5
/tmp/bao_env/bin/python -c "import torch, sklearn, psycopg2; print('torch', torch.__version__, '| sklearn', sklearn.__version__)"

In [ ]:
%%writefile /tmp/run_8_6.py
# Ablation 8.6 – độ bền của thời gian thực thi (giống notebooks/AblationStudy_RunEveryQueryNTimes.ipynb của tác giả):
# mỗi câu JOB chạy EXPLAIN (ANALYZE, VERBOSE, FORMAT JSON) N = 50 lần LIÊN TIẾP, theo thứ tự tên file
# (10a × 50, 10b × 50, …), lấy 'Execution Time'. Log cùng định dạng bao/logs/older_experiments/pg__imdb__50.txt:
#   x <lần> <ngày> <giờ> queries/join-order-benchmark/<câu>.sql <exec_ms> PG
import datetime, glob, os, sys
import psycopg2

N, TIMEOUT_MS = 50, 3 * 60 * 1000
qdir, out = sys.argv[1], sys.argv[2]
paths = sorted(glob.glob(os.path.join(qdir, "*.sql")))
with open(out, "w") as f:
    f.write(f"Read {len(paths)} queries.\nUsing Bao: False\nRunning {len(paths)} test queries...\n")
    for qi, p in enumerate(paths):
        sql = open(p).read()
        conn = psycopg2.connect("dbname=imdbload user=postgres host=localhost")
        cur = conn.cursor()
        cur.execute(f"SET statement_timeout TO {TIMEOUT_MS}")
        cur.execute("SET enable_bao TO off")
        times = []
        for i in range(N):
            try:
                cur.execute(f"EXPLAIN (ANALYZE, VERBOSE, FORMAT JSON) {sql}")
                t = cur.fetchall()[0][0][0]["Execution Time"]
            except psycopg2.errors.QueryCanceled:
                conn.rollback(); t = 2 * TIMEOUT_MS        # quy ước của tác giả: timeout ghi 2 × giới hạn
            ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")
            f.write(f"x {i} {ts} queries/join-order-benchmark/{os.path.basename(p)} {t} PG\n")
            times.append(t)
        f.flush(); conn.close()
        print(f"[{qi + 1}/{len(paths)}] {os.path.basename(p)}: lần 1 {times[0]:.0f} ms, lần 2 {times[1]:.0f} ms, "
              f"trung vị {sorted(times)[N // 2]:.0f} ms", flush=True)

In [ ]:
%%bash
# Chạy thẳng sau khi nạp IMDB, không làm nóng riêng: thí nghiệm đo chính sự khác nhau giữa các lần chạy liên tiếp
mkdir -p /kaggle/working/ablation86_logs
/tmp/bao_env/bin/python -u /tmp/run_8_6.py /tmp/lqo_ml_perspective/bao/queries/join-order-benchmark \
  /kaggle/working/ablation86_logs/pg__imdb__50.txt

In [ ]:
# Phân tích giống notebook của tác giả: chuẩn hóa theo lần chạy đầu, chênh lệch giữa lần k và k+1 (k = 1..4)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
rows = [l.split(" ") for l in open("/kaggle/working/ablation86_logs/pg__imdb__50.txt") if len(l.split(" ")) == 7]
df = pd.DataFrame({"iteration": [int(r[1]) for r in rows], "query": [r[4].split("/")[-1][:-4] for r in rows],
                   "exec_time": [float(r[5]) for r in rows]}).sort_values(["query", "iteration"])
w = df.pivot(index="query", columns="iteration", values="exec_time")
norm = w.div(w[0], axis=0)
rows = []
for k in range(1, 5):
    d = 100 * (norm[k] - norm[k - 1])
    rows.append({"giữa lần": f"{k} và {k + 1}", "trung bình (%)": d.mean(), "trung vị (%)": d.median(), "độ lệch chuẩn": d.std()})
res = pd.DataFrame(rows)
print(res.round(2).to_string(index=False))
res.to_csv("/kaggle/working/ablation86_logs/tom_tat_8_6.csv", index=False)
plt.figure(figsize=(5, 4)); plt.axhline(0, color="black", linewidth=0.2)
plt.boxplot([100 * (norm[k] - norm[k - 1]) for k in range(1, 5)], labels=[f"k = {k}" for k in range(1, 5)])
plt.ylabel("Difference in Normalized\nExecution Time [%]"); plt.xlabel("Between k-th and (k+1)-th Query Execution")
plt.tight_layout(); plt.savefig("/kaggle/working/ablation86_logs/hinh7_8_6.png", dpi=150); plt.show()